# Fourier Series Visualizer v0.2 — 원리와 코드 해설

## 1. 이 프로그램은 무엇을 하는가?

사용자가 입력한 실수 함수 f(x)를 [-L,L]에서 샘플링하고, 사인·코사인의 합으로
얼마나 가깝게 근사할 수 있는지 보여주는 학습용 데스크톱 프로그램입니다.
`실행.bat`를 더블클릭하면 브라우저 없이 독립 창으로 열립니다.
함수와 L은 입력 후 Enter로 적용하고, N과 샘플 수는 슬라이더 또는 숫자 입력으로 조절합니다.

| 화면 | 읽는 방법 |
|---|---|
| 함수 / 부분합 비교 | 검은 원래 함수와 유한한 N차 부분합을 비교 |
| 오차 그래프 | f(x)-S_N(x)가 큰 위치와 부호를 확인 |
| 계수 표 / Spectrum | 각 사인·코사인 항의 계수 및 절댓값 확인 |
| N별 수렴 오차 | 같은 표본에서 N에 따른 MSE와 RMSE 비교 |

이 프로그램은 **푸리에 급수의 계수를 수치 적분으로 계산**합니다.
푸리에 변환, DFT, FFT를 수행하는 기능은 아직 구현하지 않았습니다.
현재 |a_n|, |b_n| 그림은 개별 계수의 크기이며 amplitude spectrum 자체와는 다릅니다.

## 2. 푸리에 급수: 주기 함수를 조화파의 합으로 표현

길이가 2L인 구간을 한 주기로 보고, 그 모양이 반복된다고 생각합니다.
기본 각주파수는 ω₀=π/L이고, n번째 조화파의 각주파수는 nω₀입니다.
여기서 x는 시간일 수도 공간 좌표일 수도 있으므로 반드시 Hz로 해석하지는 않습니다.

$$f(x)\sim a_0+\sum_{n=1}^{\infty}\left[a_n\cos\left(\frac{n\pi x}{L}\right)+b_n\sin\left(\frac{n\pi x}{L}\right)\right]$$

$$a_0=\frac{1}{2L}\int_{-L}^{L}f(x)\,dx$$
$$a_n=\frac{1}{L}\int_{-L}^{L}f(x)\cos\left(\frac{n\pi x}{L}\right)dx,\quad
b_n=\frac{1}{L}\int_{-L}^{L}f(x)\sin\left(\frac{n\pi x}{L}\right)dx$$

a0는 한 주기의 평균, 즉 DC 성분입니다. **이 프로젝트는 a0/2가 아니라 a0 표기**를 사용합니다.
서로 다른 정수 차수의 사인·코사인은 [-L,L]에서 직교합니다.
함수에 해당 기저를 곱해서 적분하면 그 기저 방향의 성분만 남고, 그 결과가 계수입니다.
사인과 코사인의 제곱 적분은 L, 상수 1의 제곱 적분은 2L이므로 정규화 계수가 다릅니다.

실제 계산은 무한합 대신 n=1,…,N까지만 사용한 부분합 S_N(x)입니다.
`an[0]`은 a₁, `bn[0]`은 b₁이며 상수 a0는 배열 밖에 따로 보관합니다.

## 3. 푸리에 변환과 DFT/FFT는 무엇이 다른가?

푸리에 급수에서는 주기 때문에 가능한 조화파 주파수가 정수배로 떨어져 있습니다.
반면 비주기 신호를 다루는 푸리에 변환은 연속적인 주파수 변수 ω를 사용합니다.
다음은 각주파수를 사용하는 한 가지 정규화 관례입니다.

$$F(\omega)=\int_{-\infty}^{\infty} f(x)e^{-i\omega x}\,dx,\qquad
f(x)=\frac{1}{2\pi}\int_{-\infty}^{\infty}F(\omega)e^{i\omega x}\,d\omega.$$

위 적분의 존재와 역변환은 함수에 대한 조건이 필요합니다.
주기 함수를 변환하는 경우에는 보통 델타 분포를 사용한 선 스펙트럼으로 해석합니다.
주기를 길게 하는 관점에서 급수의 주파수 간격 π/L이 좁아지는 것으로 두 개념을 연결할 수 있습니다.

유한한 표본 y₀,…,y_{M-1}에 대한 DFT는 다음과 같습니다.

$$X_k=\sum_{j=0}^{M-1}y_j e^{-i2\pi kj/M},\quad k=0,\ldots,M-1.$$

**FFT는 DFT와 다른 변환이 아니라 DFT를 빠르게 계산하는 알고리즘**입니다.
DFT에는 M개의 표본과 M개의 frequency bin이 있으며, Fourier 급수의 절단 차수 N과는 역할이 다릅니다.
향후 FFT를 추가할 때는 양 끝점 중복, 주파수 축, 정규화, sampling frequency를 따로 설계해야 합니다.

| 구분 | 입력 관점 | 출력 관점 | 현재 구현 |
|---|---|---|---|
| Fourier Series | 주기 2L의 함수 | 정수 차수별 계수 | 구현 |
| Fourier Transform | 연속 좌표의 신호 | 연속 주파수 함수 | 설명만 제공 |
| DFT | 유한한 이산 표본 | 이산 frequency bins | 미구현 |
| FFT | DFT 계산 알고리즘 | DFT와 같은 결과 | 미구현 |

## 4. N과 샘플 수 M의 차이

N을 늘리면 근사에 사용하는 조화파 항이 많아집니다. M을 늘리면 적분과 그래프의
표본 간격 Δx=2L/(M-1)이 작아집니다. **M이 커져도 N이 같으면 추가 조화파가 생기지 않습니다.**
N이 높을수록 빠른 진동을 표현하므로 충분한 M이 필요하며, 입력 함수가 빠르게 진동하면
고정된 표본으로 계수를 정확하게 구하지 못할 수 있습니다. 표본 수를 바꿔 안정성을 확인하세요.

프로그램은 양 끝점을 포함한 `linspace`와 사다리꼴 적분을 사용합니다.

$$\int_{-L}^{L}g(x)dx\approx\sum_{j=0}^{M-2}\frac{g(x_j)+g(x_{j+1})}{2}(x_{j+1}-x_j).$$

표시 오차는 표본 오차이며 연속 구간 전체의 정확한 적분 오차나 최대 오차가 아닙니다.
$$\mathrm{MSE}=\frac1M\sum_j(f(x_j)-S_N(x_j))^2,\quad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}},\quad E_{\max}=\max_j|f(x_j)-S_N(x_j)|.$$

## 5. 대칭성과 Gibbs 현상

f(x)=x, L=π이면 기함수이므로 a0=0, a_n=0이고
$$b_n=\frac{2(-1)^{n+1}}{n}.$$
수치 적분에서는 0 대신 매우 작은 반올림 오차가 남을 수 있습니다.
짝함수 x², |x|에서는 b_n이 거의 0입니다. 대칭 자동 판별 기능은 아직 없습니다.

`sign(x)`는 0에서 불연속입니다. 불연속점 근처에서 부분합이 위아래로 진동하는 것이 Gibbs 현상입니다.
N이 커질수록 진동 영역은 좁아지지만 overshoot는 점프 높이의 약 9% 수준으로 남습니다.
좌우 극한이 -1,1이면 점프 높이는 2이므로 위쪽 peak는 약 1.18입니다.
불연속점에서는 적절한 조건 아래 좌우 극한의 평균으로 수렴합니다.
`x`나 `exp(x)`도 주기적으로 이어 붙이면 양 끝에서 불연속이 생길 수 있습니다.

## 6. 실행 및 개발 환경

현재 PC에서는 노트북 옆 `실행.bat`를 실행하세요. 노트북 커널은
`Fourier Visualizer (.venv)` 또는 `.venv/Scripts/python.exe`를 선택합니다.
노트북 자체는 소스 생성과 학습·검증용이며 전체 실행으로 GUI를 자동 실행하지 않습니다.

새 PC에서 프로젝트 폴더를 기준으로 설치하려면:

```powershell
python -m venv .venv
& ./.venv/Scripts/python.exe -m pip install -r ./fourier_visualizer/requirements.txt ipykernel
& ./.venv/Scripts/python.exe ./fourier_visualizer/desktop_app.py
```

웹 버전이 필요할 때만 `fourier_visualizer` 폴더에서 다음을 사용합니다.
```bash
pip install -r requirements.txt
streamlit run app.py
```

## 7. 파일 구조와 데이터 흐름

`입력 → 수식 검사 → 표본 생성 → 계수 적분 → 부분합 → 오차 → 그래프/표`

- `fourier_core.py`: 공통 수학 함수와 Figure 생성. Streamlit/Tkinter 상태에 의존하지 않습니다.
- `desktop_app.py`: Tkinter 창, 숫자 입력/슬라이더 동기화, 결과 표시, 이벤트 처리.
- `app.py`: 선택적으로 사용하는 Streamlit 화면. 동일한 공통 계산 함수를 가져옵니다.
- `requirements.txt`: 설치할 라이브러리. Tkinter는 Python 설치에 포함되어야 합니다.
- `실행.bat`: 현재 폴더의 가상환경으로 독립 창을 실행하는 Windows 진입점.

## 8. 최적화 내용과 한계

여러 N의 부분합은 각 N마다 1부터 다시 더하지 않고 최대 N까지 한 번 누적하여
필요한 차수의 배열만 복사합니다. 표본 수 M일 때 계산량은 O(MΣN)에서 O(M max N)으로 줄어듭니다.
스냅샷 배열은 선택 차수 개수 K에 대해 O(KM) 메모리가 필요합니다.
각 snapshot은 복사본이므로 뒤의 누적 연산에 의해 앞의 결과가 바뀌지 않습니다.

데스크톱 `FourierSession`은 마지막 함수 문자열, L, M에 대한 표본과 계수를 보관합니다.
같은 조건에서 이미 계산한 최대 차수 이하를 요청하면 적분을 생략합니다.
더 큰 차수를 요청하면 계수를 다시 계산하고, 조건이 바뀌면 새 표본으로 교체합니다.
전체 이력을 저장하지 않으므로 캐시가 무한히 커지지 않습니다.
수식 해석은 최대 32개 LRU 캐시로 반복 SymPy 해석을 줄입니다.

250ms 입력 지연 처리는 기존부터 적용된 기능입니다. 빠른 슬라이더 이동 중 예약 계산을 취소합니다.
그래프는 현재도 재생성하므로 화면 그리기 비용은 남아 있습니다.
이번 변경은 계산 중복 제거이며 전체 UI 속도가 일정 배수 빨라진다는 보장은 하지 않습니다.
큰 100×20000 삼각함수 행렬을 항상 만들지 않고 차수별 배열을 사용해 메모리 사용을 제한합니다.

## 9. 확장 계획

v0.3: 대칭 자동 판별, A_n=√(a_n²+b_n²) amplitude spectrum,
위상 관례를 정한 phase spectrum, 표본 표시, Gibbs 강조, 애니메이션, 예제 선택.
v0.4: DFT/FFT, sampling frequency, frequency bins, aliasing, 급수와 DFT 비교.
예를 들어 a cosθ+b sinθ=A cos(θ-φ) 관례라면 φ=atan2(b,a)이며 A=0의 위상은 별도 처리합니다.


## 10. 아래 코드 셀을 사용하는 방법

이후 셀은 **설명 다음에 해당 실제 코드를 배치**합니다. `module_parts.append(...)` 안의
문자열이 실제 Python 소스입니다. 각 파일의 마지막 저장 셀에서 문자열을 합쳐 파일을 생성합니다.
위에서 아래로 실행하면 최신 소스로 파일을 갱신하므로, 외부에서 수정한 파일이 있다면 먼저 보관하세요.
GUI 클래스는 메서드별로 나눠 읽도록 구성했고, 저장할 때 원래 들여쓰기로 합칩니다.


In [ ]:
from pathlib import Path
import sys
notebook_directory = Path.cwd()
if (notebook_directory / "공업수학2" / "fourier-visualizer").is_dir():
    notebook_directory /= "공업수학2/fourier-visualizer"
elif (notebook_directory / "fourier-visualizer" / "만들기.ipynb").exists():
    notebook_directory /= "fourier-visualizer"
project_directory = notebook_directory / "fourier_visualizer"
project_directory.mkdir(exist_ok=True)
print("프로젝트:", project_directory)
print("현재 Python:", sys.executable)


## `fourier_core.py` 코드 해설

아래 순서대로 코드를 모아 이 파일을 생성합니다.

In [ ]:
module_parts = []


### `모듈 준비`

라이브러리와 공통 상수를 준비합니다. 계산 모듈은 UI 상태를 읽지 않습니다.

In [ ]:
module_parts.append(r'''"""Fourier 급수의 입력 해석, 수치 계산 및 그래프 생성 공통 모듈."""

import ast

import re

import matplotlib.pyplot as plt

import numpy as np

from functools import lru_cache

import sympy as sp

X = sp.Symbol("x", real=True)

ALLOWED_NAMES = {
    "x": X, "pi": sp.pi, "E": sp.E,
    "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
    "exp": sp.exp, "log": sp.log, "sqrt": sp.sqrt,
    "abs": sp.Abs, "Abs": sp.Abs, "sign": sp.sign,
    "sinh": sp.sinh, "cosh": sp.cosh, "tanh": sp.tanh,
}

ALLOWED_NODES = (
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Call, ast.Name,
    ast.Load, ast.Constant, ast.Add, ast.Sub, ast.Mult, ast.Div,
    ast.Pow, ast.UAdd, ast.USub,
)

''')


### `parse_expression`

수식 문자열을 AST로 검사하여 허용된 연산자와 함수만 받습니다. 이후 sympify로 수식으로 변환합니다. 동일 문자열은 LRU 캐시로 재사용합니다. 허용 목록은 자원 사용량 제한까지 보장하는 샌드박스는 아닙니다.

In [ ]:
module_parts.append(r'''@lru_cache(maxsize=32)
def parse_expression(expression_text):
    """허용 목록을 검사한 문자열을 SymPy 수식으로 변환한다."""
    if not expression_text.strip() or len(expression_text) > 300:
        raise ValueError("수식을 1~300자 이내로 입력하세요.")
    # 이름, 숫자, 괄호, 기본 연산자 외에는 허용하지 않는다.
    if not re.fullmatch(r"[A-Za-z0-9_+\-*/().,\s]+", expression_text):
        raise ValueError("지원하지 않는 문자가 있습니다. 거듭제곱은 **로 입력하세요.")
    tree = ast.parse(expression_text, mode="eval")
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_NODES):
            raise ValueError("기본 산술 연산과 지원 함수만 사용할 수 있습니다.")
        if isinstance(node, ast.Name) and node.id not in ALLOWED_NAMES:
            raise ValueError(f"지원하지 않는 이름: {node.id}")
        if isinstance(node, ast.Constant):
            if type(node.value) not in (int, float):
                raise ValueError("숫자 상수만 사용할 수 있습니다.")
        if isinstance(node, ast.Call):
            if (not isinstance(node.func, ast.Name)
                    or node.func.id not in ALLOWED_NAMES
                    or not callable(ALLOWED_NAMES[node.func.id])
                    or len(node.args) != 1 or node.keywords):
                raise ValueError("지원 함수에는 인자 하나만 입력하세요.")
    expression = sp.sympify(expression_text, locals=ALLOWED_NAMES)
    if not isinstance(expression, sp.Expr) or expression.free_symbols - {X}:
        raise ValueError("x에 대한 실수 함수를 입력하세요.")
    return expression

''')


### `sample_function`

lambdify로 NumPy 함수를 만들고 표본 x에 적용합니다. 상수 함수가 scalar를 반환하면 broadcast_to로 길이를 맞춥니다. 복소수와 NaN/inf를 거부합니다. 표본 사이의 특이점까지 모두 검출하는 것은 아닙니다.

In [ ]:
module_parts.append(r'''def sample_function(expression, x):
    """상수 함수도 x와 같은 크기로 확장하고 비유한/복소 값을 거부한다."""
    numpy_function = sp.lambdify(X, expression, modules="numpy")
    with np.errstate(all="ignore"):
        values = np.asarray(numpy_function(x))
    if np.iscomplexobj(values):
        raise ValueError("복소수 값이 발생했습니다. 실수 함수를 입력하세요.")
    values = np.array(np.broadcast_to(values, x.shape), dtype=float, copy=True)
    if not np.all(np.isfinite(values)):
        raise ValueError("샘플에서 NaN 또는 inf가 발생했습니다. 함수와 구간을 확인하세요.")
    return values

''')


### `calculate_fourier_coefficients`

표본의 크기, 유한성, 정렬, 끝점을 검사합니다. 기본 각도 πx/L은 한 번 만들고 각 n에 대해 곱합니다. trapezoid로 상수항/코사인/사인 계수를 적분합니다. 시간 복잡도 O(MN), 작업 배열은 O(M)입니다.

In [ ]:
module_parts.append(r'''def calculate_fourier_coefficients(x, y, L, max_N):
    """[-L,L] 표본에 사다리꼴 적분을 적용한다. an[0]은 a_1이다."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if not np.isfinite(L) or L <= 0:
        raise ValueError("L은 유한한 양수여야 합니다.")
    if not isinstance(max_N, (int, np.integer)) or max_N < 1:
        raise ValueError("max_N은 양의 정수여야 합니다.")
    if x.ndim != 1 or x.size < 2 or x.shape != y.shape:
        raise ValueError("x와 y는 길이가 같은 1차원 표본이어야 합니다.")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError("표본은 모두 유한해야 합니다.")
    if not np.all(np.diff(x) > 0):
        raise ValueError("x는 오름차순이어야 합니다.")
    if not np.isclose(x[0], -L) or not np.isclose(x[-1], L):
        raise ValueError("표본 구간은 [-L, L]이어야 합니다.")

    # NumPy 2.x의 API를 우선 사용하고 이전 버전도 지원한다.
    integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    cosine_coefficients = np.zeros(max_N)
    sine_coefficients = np.zeros(max_N)
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        a0 = float(integrate(y, x=x) / (2 * L))
        base_angle = np.pi * (x / L)
        for n in range(1, max_N + 1):
            angle = n * base_angle
            cosine_coefficients[n - 1] = integrate(y * np.cos(angle), x=x) / L
            sine_coefficients[n - 1] = integrate(y * np.sin(angle), x=x) / L
    if not np.all(np.isfinite(np.r_[a0, cosine_coefficients, sine_coefficients])):
        raise ValueError("계수가 유한하지 않습니다. 함수의 크기를 줄여 주세요.")
    return a0, cosine_coefficients, sine_coefficients

''')


### `calculate_fourier_sum`

기존 단일 N 함수 인터페이스를 유지합니다. 여러 부분합 계산 함수에 N 하나를 전달하므로 공식 구현을 중복하지 않습니다.

In [ ]:
module_parts.append(r'''def calculate_fourier_sum(x, L, a0, an, bn, N):
    """a0 + Σ(an cos(nπx/L) + bn sin(nπx/L))를 계산한다."""
    return calculate_fourier_sums(x, L, a0, an, bn, [N])[N]


''')


### `calculate_fourier_sums`

요청 차수를 집합으로 중복 제거하고 최대 차수까지 누적합니다. 필요한 순간 copy()로 별도 결과를 저장합니다. 빈 요청은 빈 사전, N=0은 상수항을 반환합니다.

In [ ]:
module_parts.append(r'''def calculate_fourier_sums(x, L, a0, an, bn, orders):
    """한 번 누적하여 여러 차수의 부분합을 만든다. 필요한 차수만 복사한다."""
    if not np.isfinite(L) or L <= 0:
        raise ValueError("L은 유한한 양수여야 합니다.")
    orders = set(orders)
    if any(not isinstance(n, (int, np.integer)) or not 0 <= n <= min(len(an), len(bn)) for n in orders):
        raise ValueError("N에 필요한 Fourier 계수가 부족합니다.")
    if not orders:
        return {}
    x = np.asarray(x, dtype=float)
    result = np.full_like(x, a0, dtype=float)
    snapshots = {0: result.copy()} if 0 in orders else {}
    with np.errstate(over="raise", invalid="raise"):
        base_angle = np.pi * (x / L)
        for n in range(1, max(orders) + 1):
            angle = n * base_angle
            result += an[n - 1] * np.cos(angle) + bn[n - 1] * np.sin(angle)
            if n in orders:
                snapshots[n] = result.copy()
    if not np.all(np.isfinite(result)):
        raise ValueError("부분합에서 비유한 값이 발생했습니다.")
    return snapshots


''')


### `FourierSession`

GUI가 보유하는 계산 캐시입니다. 동일 함수/L/M이며 충분한 계수가 있으면 배열을 잘라 재사용합니다. 실패한 계산은 캐시에 저장하지 않습니다. 반환 배열은 읽기 전용으로 취급하고 외부에서 수정하지 않습니다.

In [ ]:
module_parts.append(r'''class FourierSession:
    """마지막 함수/구간/표본의 계수를 보관하는 작은 데스크톱 전용 캐시."""

    def __init__(self):
        self.key = None
        self.data = None

    def prepare(self, expression_text, L, num_points, max_N):
        key = (expression_text.strip(), L, num_points)
        if key == self.key and len(self.data[4]) >= max_N:
            expression, x, y, a0, an, bn = self.data
            return expression, x, y, a0, an[:max_N], bn[:max_N]
        expression = parse_expression(key[0])
        if key == self.key:
            x, y = self.data[1:3]
        else:
            x = np.linspace(-L, L, num_points)
            y = sample_function(expression, x)
        a0, an, bn = calculate_fourier_coefficients(x, y, L, max_N)
        # 계산에 성공한 결과만 교체한다. 잘못된 입력은 기존 캐시를 오염시키지 않는다.
        self.key = key
        self.data = (expression, x, y, a0, an, bn)
        return self.data

''')


### `calculate_errors`

같은 표본의 원래 값과 부분합 차이를 계산합니다. MSE는 제곱 오차 평균, RMSE는 그 제곱근, Maximum Absolute Error는 표본 절댓값 오차의 최대입니다.

In [ ]:
module_parts.append(r'''def calculate_errors(y, approximation):
    """같은 표본에서의 오차: 연속 구간 전체의 최대 오차와는 구별된다."""
    with np.errstate(over="raise", invalid="raise"):
        error = y - approximation
        mse = float(np.mean(error ** 2))
        rmse = float(np.sqrt(mse))
        maximum_error = float(np.max(np.abs(error)))
    return error, {"MSE": mse, "RMSE": rmse, "Maximum Absolute Error": maximum_error}

''')


### `make_comparison_figure`

원래 함수와 N별 부분합을 같은 좌표축에 그립니다. Figure만 반환하므로 웹/데스크톱에서 공통으로 표시할 수 있습니다.

In [ ]:
module_parts.append(r'''def make_comparison_figure(x, y, approximations, title):
    figure, axes = plt.subplots(figsize=(7, 4))
    axes.plot(x, y, color="black", linewidth=2, label="f(x)")
    for order, approximation in approximations.items():
        axes.plot(x, approximation, linewidth=1.4, label=f"N = {order}")
    axes.set(title=title, xlabel="x", ylabel="Function value")
    axes.grid(True, alpha=0.3)
    axes.legend()
    figure.tight_layout()
    return figure

''')


### `make_error_figure`

오차의 위치와 부호를 보여줍니다. 0 기준선을 함께 표시하므로 어느 구간에서 근사값이 큰지 작은지 확인할 수 있습니다.

In [ ]:
module_parts.append(r'''def make_error_figure(x, error):
    figure, axes = plt.subplots(figsize=(12, 3))
    axes.plot(x, error, label="f(x) - S_N(x)", color="tab:red")
    axes.axhline(0, color="gray", linewidth=0.8)
    axes.set(title="Approximation error", xlabel="x", ylabel="Error")
    axes.grid(True, alpha=0.3)
    axes.legend()
    figure.tight_layout()
    return figure

''')


### `make_spectrum_figure`

정수 차수 n에서 |a_n|과 |b_n|을 stem plot으로 표시합니다. 두 계수의 절댓값이며 합성 amplitude나 FFT 결과가 아닙니다.

In [ ]:
module_parts.append(r'''def make_spectrum_figure(an, bn):
    orders = np.arange(1, len(an) + 1)
    figure, axes = plt.subplots(figsize=(10, 3.5))
    axes.stem(orders, np.abs(an), linefmt="C0-", markerfmt="C0o",
              basefmt=" ", label="|a_n|")
    axes.stem(orders, np.abs(bn), linefmt="C1--", markerfmt="C1x",
              basefmt=" ", label="|b_n|")
    axes.set(title="Coefficient spectrum", xlabel="n", ylabel="Coefficient magnitude")
    axes.grid(True, alpha=0.3)
    axes.legend()
    figure.tight_layout()
    return figure
''')


In [ ]:
(project_directory / 'fourier_core.py').write_text(''.join(module_parts), encoding='utf-8')


## `desktop_app.py` 코드 해설

아래 순서대로 코드를 모아 이 파일을 생성합니다.

In [ ]:
module_parts = []


### `모듈 준비`

라이브러리와 공통 상수를 준비합니다. 계산 모듈은 UI 상태를 읽지 않습니다.

In [ ]:
module_parts.append(r'''"""브라우저 없이 실행되는 Tkinter Fourier 학습 프로그램."""

import tkinter as tk
from tkinter import ttk

import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
import numpy as np

from fourier_core import (
    parse_expression, FourierSession, calculate_fourier_sums,
    calculate_errors, make_comparison_figure,
    make_error_figure, make_spectrum_figure,
)


''')


### `FourierDesktop`

데스크톱 화면을 관리하는 클래스입니다.

In [ ]:
module_parts.append(r'''class FourierDesktop(tk.Tk):
    """입력 상태와 화면을 관리하며 계산은 fourier_core에 위임한다."""

''')


### `__init__`

Tk 변수, 계산 세션, 입력 변경 추적, 초기 계산 예약을 설정합니다. trace_add는 숫자 입력과 슬라이더의 상태 연결을 담당합니다.

In [ ]:
module_parts.append(r'''    def __init__(self):
        super().__init__()
        self.title("Fourier Series Visualizer v0.2 — Desktop")
        self.geometry("1400x900")
        self.minsize(1050, 700)
        self.protocol("WM_DELETE_WINDOW", self.close)
        self.figures = []
        self.session = FourierSession()
        self.pending_update = None
        self.expression = tk.StringVar(value="x")
        self.length = tk.StringVar(value="pi")
        self.order = tk.IntVar(value=10)
        self.order_text = tk.StringVar(value="10")
        self.points = tk.IntVar(value=5000)
        self.points_text = tk.StringVar(value="5000")
        self.status = tk.StringVar(value="")
        self.information = tk.StringVar(value="")
        self.comparisons = {n: tk.BooleanVar(value=n in [1, 3, 5, 10]) for n in [1, 3, 5, 10, 20, 50]}
        self.metric_values = [tk.StringVar(value="—") for _ in range(3)]
        self.build_ui()
        self.order.trace_add("write", self.sync_order_input)
        self.order_text.trace_add("write", self.apply_order_input)
        self.points.trace_add("write", self.sync_points_input)
        self.points_text.trace_add("write", self.apply_points_input)
        self.bind("<Return>", lambda event: self.recalculate())
        self.after(100, self.recalculate)

    # ── 입력 패널과 결과 탭 ───────────────────────────────────────────
''')


### `build_ui`

왼쪽 입력 패널과 오른쪽 결과 탭을 만듭니다. N은 1~100, 샘플 수는 500~20000입니다. Entry와 Scale은 서로 다른 편집/확정 변수를 연결합니다.

In [ ]:
module_parts.append(r'''    def build_ui(self):
        sidebar = ttk.Frame(self, padding=16, width=245)
        sidebar.pack(side="left", fill="y")
        ttk.Label(sidebar, text="Fourier Series", font=("맑은 고딕", 17, "bold")).pack(anchor="w")
        ttk.Label(sidebar, text="데스크톱 시각화 v0.2").pack(anchor="w", pady=(0, 20))
        for label, variable in [("함수 f(x)", self.expression), ("구간의 반길이 L", self.length)]:
            ttk.Label(sidebar, text=label).pack(anchor="w", pady=(10, 3))
            ttk.Entry(sidebar, textvariable=variable, width=25).pack(fill="x")
        ttk.Label(sidebar, text="x, x**2, sin(x), cos(x)\nexp(x), abs(x), sign(x), 3").pack(anchor="w", pady=10)
        ttk.Label(sidebar, text="현재 Fourier 차수 N").pack(anchor="w")
        # 편집 중 빈 문자열은 허용하고, 실제 차수는 1~100의 정수로 유지한다.
        self.order_input = ttk.Spinbox(
            sidebar, from_=1, to=100, increment=1, width=8,
            textvariable=self.order_text, validate="key",
            validatecommand=(self.register(self.validate_order_input), "%P"),
        )
        self.order_input.pack(anchor="w", pady=(4, 0))
        self.order_input.bind("<FocusOut>", self.finish_order_input)
        tk.Scale(sidebar, from_=1, to=100, orient="horizontal", variable=self.order,
                 command=self.schedule_update).pack(fill="x")
        ttk.Label(sidebar, text="샘플링 점 개수").pack(anchor="w", pady=(10, 0))
        self.points_input = ttk.Spinbox(
            sidebar, from_=500, to=20000, increment=1, width=8,
            textvariable=self.points_text, validate="key",
            validatecommand=(self.register(self.validate_points_input), "%P"),
        )
        self.points_input.pack(anchor="w", pady=(4, 0))
        self.points_input.bind("<FocusOut>", self.finish_points_input)
        tk.Scale(sidebar, from_=500, to=20000, resolution=1, orient="horizontal",
                 variable=self.points, command=self.schedule_update).pack(fill="x")
        ttk.Label(sidebar, text="비교할 N (모두 해제 가능)").pack(anchor="w", pady=(15, 5))
        for n, variable in self.comparisons.items():
            ttk.Checkbutton(sidebar, text=f"N = {n}", variable=variable,
                            command=self.schedule_update).pack(anchor="w")
        ttk.Button(sidebar, text="계산 / 그래프 갱신", command=self.recalculate).pack(fill="x", pady=20)
        ttk.Label(sidebar, text="함수와 L 변경 후 Enter 또는\n계산 버튼을 누르세요.\n\nsign(x)에서 N을 늘려\nGibbs 현상을 확인하세요.").pack(anchor="w")

        content = ttk.Frame(self, padding=12)
        content.pack(side="left", fill="both", expand=True)
        ttk.Label(content, text="f(x) ≈ a₀ + Σ [aₙ cos(nπx/L) + bₙ sin(nπx/L)]",
                  font=("맑은 고딕", 13)).pack(anchor="w", pady=(0, 8))
        metric_frame = ttk.Frame(content)
        metric_frame.pack(fill="x", pady=8)
        for index, label in enumerate(["MSE", "RMSE", "Maximum Absolute Error"]):
            frame = ttk.LabelFrame(metric_frame, text=label, padding=10)
            frame.pack(side="left", fill="x", expand=True, padx=4)
            ttk.Label(frame, textvariable=self.metric_values[index], font=("Segoe UI", 17)).pack()
        tk.Label(content, textvariable=self.status, fg="#b42318", anchor="w", wraplength=850).pack(fill="x", pady=5)
        self.tabs = ttk.Notebook(content)
        self.tabs.pack(fill="both", expand=True)
        self.plot_tab = ttk.Frame(self.tabs)
        self.error_tab = ttk.Frame(self.tabs)
        self.coefficient_tab = ttk.Frame(self.tabs)
        self.convergence_tab = ttk.Frame(self.tabs)
        for frame, title in [(self.plot_tab, "함수 / 부분합 비교"), (self.error_tab, "오차 그래프"),
                             (self.coefficient_tab, "계수 표 / Spectrum"), (self.convergence_tab, "N별 수렴 오차")]:
            self.tabs.add(frame, text=title)
        ttk.Label(content, textvariable=self.information, wraplength=900).pack(anchor="w", pady=10)
        ttk.Label(content, text="오차는 표본 기준입니다. 불연속점과 주기 경계에서 최대 오차가 0으로 수렴하지 않을 수 있습니다.").pack(anchor="w")

''')


### `validate_order_input`

편집 중 빈 문자열은 허용하되 숫자는 1~100 정수로 제한합니다. 텍스트 검증 단계에서 부호/소수/문자를 막습니다.

In [ ]:
module_parts.append(r'''    @staticmethod
    def validate_order_input(text):
        return text == "" or (text.isascii() and text.isdigit() and 1 <= int(text) <= 100)

''')


### `apply_order_input`

유효한 입력을 정수 N 상태에 적용하고 계산을 예약합니다. 값이 같으면 다시 설정하지 않아 상호 콜백 반복을 줄입니다.

In [ ]:
module_parts.append(r'''    def apply_order_input(self, *_):
        text = self.order_text.get()
        if text and self.validate_order_input(text):
            value = int(text)
            if self.order.get() != value:
                self.order.set(value)
                self.schedule_update()

''')


### `sync_order_input`

슬라이더의 정수 N을 숫자 입력 문자열로 반영합니다.

In [ ]:
module_parts.append(r'''    def sync_order_input(self, *_):
        value = str(self.order.get())
        if self.order_text.get() != value:
            self.order_text.set(value)

''')


### `finish_order_input`

편집 종료 시 숫자 입력을 마지막 유효 N으로 정규화합니다.

In [ ]:
module_parts.append(r'''    def finish_order_input(self, *_):
        self.sync_order_input()

''')


### `schedule_update`

이전 after 예약을 취소하고 250ms 뒤 계산을 예약합니다. 빠른 연속 입력을 모아서 계산합니다.

In [ ]:
module_parts.append(r'''    def schedule_update(self, *_):
        if self.pending_update is not None:
            self.after_cancel(self.pending_update)
        self.pending_update = self.after(250, self.recalculate)

''')


### `validate_points_input`

500을 타이핑할 때 중간 문자열 5, 50을 허용합니다. 20000 초과, 소수, 음수, 문자는 허용하지 않습니다.

In [ ]:
module_parts.append(r'''    @staticmethod
    def validate_points_input(text):
        # 500을 입력하는 중의 '5', '50'도 허용하되 계산에는 적용하지 않는다.
        return text == "" or (
            text.isascii() and text.isdigit() and len(text) <= 5 and int(text) <= 20000
        )

''')


### `apply_points_input`

중간 편집값은 실제 계산에 쓰지 않습니다. 500~20000의 유효한 정수에만 샘플 수를 바꾸고 갱신을 예약합니다.

In [ ]:
module_parts.append(r'''    def apply_points_input(self, *_):
        text = self.points_text.get()
        if text and self.validate_points_input(text) and 500 <= int(text) <= 20000:
            value = int(text)
            if self.points.get() != value:
                self.points.set(value)
                self.schedule_update()

''')


### `sync_points_input`

슬라이더 표본 수를 숫자 입력칸에 반영합니다. 슬라이더 단위도 1이므로 1234 같은 정수가 보존됩니다.

In [ ]:
module_parts.append(r'''    def sync_points_input(self, *_):
        value = str(self.points.get())
        if self.points_text.get() != value:
            self.points_text.set(value)

''')


### `finish_points_input`

빈 값 또는 500 미만으로 편집을 끝내면 마지막 유효한 값으로 복원합니다.

In [ ]:
module_parts.append(r'''    def finish_points_input(self, *_):
        # 빈 값 또는 500 미만으로 편집을 마치면 마지막 유효값으로 복원한다.
        self.sync_points_input()

    # ── 그래프 및 스크롤 가능한 표 공통 처리 ──────────────────────────
''')


### `embed_figure`

Matplotlib Figure를 Tk 캔버스에 연결하고 확대/이동/저장 툴바를 붙입니다.

In [ ]:
module_parts.append(r'''    def embed_figure(self, parent, figure):
        self.figures.append(figure)
        canvas = FigureCanvasTkAgg(figure, master=parent)
        toolbar = NavigationToolbar2Tk(canvas, parent, pack_toolbar=False)
        toolbar.update()
        toolbar.pack(side="bottom", fill="x")
        canvas.get_tk_widget().pack(fill="both", expand=True)
        canvas.draw()

''')


### `table`

열 이름과 행 데이터를 받아 스크롤 가능한 Treeview 표를 만듭니다. 계수 표와 오차 표에서 공통 사용합니다.

In [ ]:
module_parts.append(r'''    @staticmethod
    def table(parent, columns, rows):
        frame = ttk.Frame(parent)
        frame.pack(fill="both", expand=True)
        tree = ttk.Treeview(frame, columns=columns, show="headings", height=9)
        scrollbar = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
        tree.configure(yscrollcommand=scrollbar.set)
        for column in columns:
            tree.heading(column, text=column)
            tree.column(column, width=120, anchor="center")
        for row in rows:
            tree.insert("", "end", values=[f"{v:.8g}" if isinstance(v, (float, np.floating)) else v for v in row])
        scrollbar.pack(side="right", fill="y")
        tree.pack(fill="both", expand=True)

''')


### `clear_results`

이전 위젯을 제거하고 pyplot Figure를 닫아 반복 갱신 시 자원 누적을 방지합니다.

In [ ]:
module_parts.append(r'''    def clear_results(self):
        for tab in [self.plot_tab, self.error_tab, self.coefficient_tab, self.convergence_tab]:
            for widget in tab.winfo_children():
                widget.destroy()
        for figure in self.figures:
            plt.close(figure)
        self.figures.clear()

    # ── 수치 계산과 화면 갱신 ─────────────────────────────────────────
''')


### `recalculate`

입력을 확정하고 L 검증 → 캐시 준비 → 다중 부분합 → 오차 계산을 수행합니다. 오류 시 이전 결과를 지우고 원인을 표시합니다. 성공 시 네 그래프와 표/정보를 갱신합니다.

In [ ]:
module_parts.append(r'''    def recalculate(self):
        self.finish_order_input()
        self.finish_points_input()
        if self.pending_update is not None:
            self.after_cancel(self.pending_update)
            self.pending_update = None
        try:
            L = float(parse_expression(self.length.get()))
            if not np.isfinite(L) or L <= 0 or not np.isfinite(2 * L):
                raise ValueError("L과 주기 2L은 유한한 양수여야 합니다.")
            N = self.order.get()
            selected = [n for n, value in self.comparisons.items() if value.get()]
            expression, x, y, a0, an, bn = self.session.prepare(
                self.expression.get(), L, self.points.get(), max([N, *selected])
            )
            sums = calculate_fourier_sums(x, L, a0, an, bn, [N, *selected])
            error, metrics = calculate_errors(y, sums[N])
            convergence = []
            for n in selected:
                _, values = calculate_errors(y, sums[n])
                convergence.append([n, values["MSE"], values["RMSE"]])
        except Exception as exc:
            self.status.set(f"입력 오류: {exc}")
            self.clear_results()
            for value in self.metric_values:
                value.set("—")
            self.information.set("함수와 L을 수정한 뒤 다시 계산하세요.")
            return

        self.status.set("")
        self.clear_results()
        for variable, value in zip(self.metric_values, metrics.values()):
            variable.set(f"{value:.6g}")
        self.plot_tab.columnconfigure((0, 1), weight=1, uniform="charts")
        self.plot_tab.rowconfigure(0, weight=1)
        for index, (title, data) in enumerate([
            (f"Current approximation: N={N}", {N: sums[N]}),
            ("Convergence comparison", {n: sums[n] for n in selected}),
        ]):
            frame = ttk.Frame(self.plot_tab)
            frame.grid(row=0, column=index, sticky="nsew")
            self.embed_figure(frame, make_comparison_figure(x, y, data, title))
        self.embed_figure(self.error_tab, make_error_figure(x, error))
        ttk.Label(self.coefficient_tab, text=f"a₀ = {a0:.12g}", padding=8).pack(anchor="w")
        spectrum = ttk.Frame(self.coefficient_tab, height=280)
        spectrum.pack(fill="both", expand=True)
        self.embed_figure(spectrum, make_spectrum_figure(an, bn))
        self.table(self.coefficient_tab, ["n", "a_n", "b_n", "|a_n|", "|b_n|"],
                   [[n, a, b, abs(a), abs(b)] for n, (a, b) in enumerate(zip(an, bn), start=1)])
        self.table(self.convergence_tab, ["N", "MSE", "RMSE"], convergence)
        self.information.set(f"f(x) = {expression}    구간: [{-L:.6g}, {L:.6g}]    period = 2L = {2*L:.6g}    current N = {N}")

''')


### `close`

예약 작업을 취소하고 Figure와 창을 닫습니다.

In [ ]:
module_parts.append(r'''    def close(self):
        if self.pending_update is not None:
            self.after_cancel(self.pending_update)
        self.clear_results()
        self.destroy()


if __name__ == "__main__":
    FourierDesktop().mainloop()
''')


In [ ]:
(project_directory / 'desktop_app.py').write_text(''.join(module_parts), encoding='utf-8')


## `app.py` 코드 해설

아래 순서대로 코드를 모아 이 파일을 생성합니다.

In [ ]:
module_parts = []


### `모듈 준비`

라이브러리와 공통 상수를 준비합니다. 계산 모듈은 UI 상태를 읽지 않습니다.

In [ ]:
module_parts.append(r'''"""선택적으로 실행하는 Streamlit 화면. 공통 계산은 fourier_core에 있다."""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st
import sympy as sp
from fourier_core import (
    parse_expression, sample_function, calculate_fourier_coefficients,
    calculate_fourier_sum, calculate_fourier_sums, calculate_errors,
    make_comparison_figure, make_error_figure, make_spectrum_figure,
)


''')


### `show_figure`

웹 화면에 Figure를 표시한 뒤 finally에서 닫습니다.

In [ ]:
module_parts.append(r'''def show_figure(figure):
    """표시 후 Figure를 닫아 Streamlit 재실행 시 메모리 누적을 막는다."""
    try:
        st.pyplot(figure)
    finally:
        plt.close(figure)

''')


### `main`

Streamlit sidebar 입력, 공통 수학 함수 호출, 두 열 그래프와 표를 구성합니다. 데스크톱 실행에는 이 함수가 필요하지 않습니다.

In [ ]:
module_parts.append(r'''def main():
    st.set_page_config(page_title="Fourier Series Visualizer v0.2", layout="wide")
    st.title("Fourier Series Visualizer v0.2")
    st.write("[-L, L]에서 정의된 함수의 주기적 확장과 Fourier 부분합을 살펴봅니다.")
    st.latex(r"f(x) \sim a_0 + \sum_{n=1}^{\infty}\left[a_n\cos\frac{n\pi x}{L}+b_n\sin\frac{n\pi x}{L}\right]")

    with st.sidebar:
        st.header("함수 및 계산 설정")
        expression_text = st.text_input("f(x)", value="x")
        st.caption("예: x, x**2, sin(x), cos(x), exp(x), abs(x), sign(x), 3")
        length_text = st.text_input("L (양수, 기본값 pi)", value="pi")
        current_N = st.slider("현재 Fourier 차수 N", 1, 100, 10)
        num_points = st.slider("샘플링 점 개수", 500, 20000, 5000, step=100)
        comparison_N = st.multiselect("비교할 N", [1, 3, 5, 10, 20, 50], default=[1, 3, 5, 10])

    try:
        expression = parse_expression(expression_text)
        length_expression = parse_expression(length_text)
        if length_expression.free_symbols:
            raise ValueError("L에는 x가 없는 양수 상수를 입력하세요.")
        L = float(length_expression)
        if not np.isfinite(L) or L <= 0 or not np.isfinite(2 * L):
            raise ValueError("L 및 주기 2L은 유한한 양수여야 합니다.")
        x = np.linspace(-L, L, num_points)
        y = sample_function(expression, x)
        # 빈 비교 목록도 안전하고 현재 N이 더 커도 충분히 계산한다.
        max_N = max([current_N, *comparison_N])
        a0, an, bn = calculate_fourier_coefficients(x, y, L, max_N)
        orders = sorted(set([current_N, *comparison_N]))
        approximations = calculate_fourier_sums(x, L, a0, an, bn, orders)
        error, metrics = calculate_errors(y, approximations[current_N])
        convergence_rows = []
        for order in sorted(comparison_N):
            _, order_metrics = calculate_errors(y, approximations[order])
            convergence_rows.append({"N": order, "MSE": order_metrics["MSE"], "RMSE": order_metrics["RMSE"]})
    except Exception as exc:
        st.error(f"입력 또는 수치 계산을 확인하세요: {exc}")
        return

    left, right = st.columns(2)
    with left:
        st.subheader(f"현재 N = {current_N}의 Fourier approximation")
        show_figure(make_comparison_figure(x, y, {current_N: approximations[current_N]}, "Current approximation"))
    with right:
        st.subheader("여러 N의 convergence comparison")
        if not comparison_N:
            st.info("비교할 N을 선택하면 부분합이 추가됩니다.")
        show_figure(make_comparison_figure(x, y, {n: approximations[n] for n in sorted(comparison_N)}, "Convergence comparison"))

    for column, (name, value) in zip(st.columns(3), metrics.items()):
        column.metric(name, f"{value:.6g}")
    st.caption("오차는 양 끝점을 포함한 표본에서 계산합니다. 불연속점의 값과 주기 경계 때문에 최대 오차가 0으로 수렴하지 않을 수 있습니다.")
    st.subheader("Error graph")
    show_figure(make_error_figure(x, error))

    st.subheader("Fourier coefficient table")
    st.write(f"a0 = {a0:.12g}")
    st.caption(f"현재 N과 비교 N에 필요한 계수 n = 1, …, {max_N}을 표시합니다.")
    coefficient_table = pd.DataFrame({
        "n": np.arange(1, max_N + 1), "a_n": an, "b_n": bn,
        "|a_n|": np.abs(an), "|b_n|": np.abs(bn),
    })
    st.dataframe(coefficient_table, hide_index=True)
    st.subheader("Coefficient spectrum")
    show_figure(make_spectrum_figure(an, bn))

    st.subheader("N별 수렴 오차 비교")
    st.dataframe(pd.DataFrame(convergence_rows, columns=["N", "MSE", "RMSE"]), hide_index=True)
    st.caption("sign(x)를 입력하고 N을 증가시키면 x = 0 및 주기 경계 근처의 Gibbs 진동을 관찰할 수 있습니다. 진동 영역은 좁아지지만 overshoot는 남습니다.")

    st.divider()
    st.subheader("현재 함수 정보")
    st.latex("f(x) = " + sp.latex(expression))
    st.write(f"구간: [{-L:.8g}, {L:.8g}] · period = 2L = {2 * L:.8g} · current N = {current_N}")

if __name__ == "__main__":
    main()
''')


In [ ]:
(project_directory / 'app.py').write_text(''.join(module_parts), encoding='utf-8')


## 부속 파일 저장

의존성 목록과 실행 파일, README도 함께 재생성합니다.

In [ ]:
file_text = r'''streamlit>=1.32
numpy>=2.0
matplotlib>=3.9
pandas>=2.2.2
sympy>=1.13
'''
(project_directory / 'requirements.txt').write_text(file_text, encoding='utf-8')


In [ ]:
file_text = r'''# Fourier Series Visualizer v0.2 — 원리와 코드 해설

## 1. 이 프로그램은 무엇을 하는가?

사용자가 입력한 실수 함수 f(x)를 [-L,L]에서 샘플링하고, 사인·코사인의 합으로
얼마나 가깝게 근사할 수 있는지 보여주는 학습용 데스크톱 프로그램입니다.
`실행.bat`를 더블클릭하면 브라우저 없이 독립 창으로 열립니다.
함수와 L은 입력 후 Enter로 적용하고, N과 샘플 수는 슬라이더 또는 숫자 입력으로 조절합니다.

| 화면 | 읽는 방법 |
|---|---|
| 함수 / 부분합 비교 | 검은 원래 함수와 유한한 N차 부분합을 비교 |
| 오차 그래프 | f(x)-S_N(x)가 큰 위치와 부호를 확인 |
| 계수 표 / Spectrum | 각 사인·코사인 항의 계수 및 절댓값 확인 |
| N별 수렴 오차 | 같은 표본에서 N에 따른 MSE와 RMSE 비교 |

이 프로그램은 **푸리에 급수의 계수를 수치 적분으로 계산**합니다.
푸리에 변환, DFT, FFT를 수행하는 기능은 아직 구현하지 않았습니다.
현재 |a_n|, |b_n| 그림은 개별 계수의 크기이며 amplitude spectrum 자체와는 다릅니다.

## 2. 푸리에 급수: 주기 함수를 조화파의 합으로 표현

길이가 2L인 구간을 한 주기로 보고, 그 모양이 반복된다고 생각합니다.
기본 각주파수는 ω₀=π/L이고, n번째 조화파의 각주파수는 nω₀입니다.
여기서 x는 시간일 수도 공간 좌표일 수도 있으므로 반드시 Hz로 해석하지는 않습니다.

$$f(x)\sim a_0+\sum_{n=1}^{\infty}\left[a_n\cos\left(\frac{n\pi x}{L}\right)+b_n\sin\left(\frac{n\pi x}{L}\right)\right]$$

$$a_0=\frac{1}{2L}\int_{-L}^{L}f(x)\,dx$$
$$a_n=\frac{1}{L}\int_{-L}^{L}f(x)\cos\left(\frac{n\pi x}{L}\right)dx,\quad
b_n=\frac{1}{L}\int_{-L}^{L}f(x)\sin\left(\frac{n\pi x}{L}\right)dx$$

a0는 한 주기의 평균, 즉 DC 성분입니다. **이 프로젝트는 a0/2가 아니라 a0 표기**를 사용합니다.
서로 다른 정수 차수의 사인·코사인은 [-L,L]에서 직교합니다.
함수에 해당 기저를 곱해서 적분하면 그 기저 방향의 성분만 남고, 그 결과가 계수입니다.
사인과 코사인의 제곱 적분은 L, 상수 1의 제곱 적분은 2L이므로 정규화 계수가 다릅니다.

실제 계산은 무한합 대신 n=1,…,N까지만 사용한 부분합 S_N(x)입니다.
`an[0]`은 a₁, `bn[0]`은 b₁이며 상수 a0는 배열 밖에 따로 보관합니다.

## 3. 푸리에 변환과 DFT/FFT는 무엇이 다른가?

푸리에 급수에서는 주기 때문에 가능한 조화파 주파수가 정수배로 떨어져 있습니다.
반면 비주기 신호를 다루는 푸리에 변환은 연속적인 주파수 변수 ω를 사용합니다.
다음은 각주파수를 사용하는 한 가지 정규화 관례입니다.

$$F(\omega)=\int_{-\infty}^{\infty} f(x)e^{-i\omega x}\,dx,\qquad
f(x)=\frac{1}{2\pi}\int_{-\infty}^{\infty}F(\omega)e^{i\omega x}\,d\omega.$$

위 적분의 존재와 역변환은 함수에 대한 조건이 필요합니다.
주기 함수를 변환하는 경우에는 보통 델타 분포를 사용한 선 스펙트럼으로 해석합니다.
주기를 길게 하는 관점에서 급수의 주파수 간격 π/L이 좁아지는 것으로 두 개념을 연결할 수 있습니다.

유한한 표본 y₀,…,y_{M-1}에 대한 DFT는 다음과 같습니다.

$$X_k=\sum_{j=0}^{M-1}y_j e^{-i2\pi kj/M},\quad k=0,\ldots,M-1.$$

**FFT는 DFT와 다른 변환이 아니라 DFT를 빠르게 계산하는 알고리즘**입니다.
DFT에는 M개의 표본과 M개의 frequency bin이 있으며, Fourier 급수의 절단 차수 N과는 역할이 다릅니다.
향후 FFT를 추가할 때는 양 끝점 중복, 주파수 축, 정규화, sampling frequency를 따로 설계해야 합니다.

| 구분 | 입력 관점 | 출력 관점 | 현재 구현 |
|---|---|---|---|
| Fourier Series | 주기 2L의 함수 | 정수 차수별 계수 | 구현 |
| Fourier Transform | 연속 좌표의 신호 | 연속 주파수 함수 | 설명만 제공 |
| DFT | 유한한 이산 표본 | 이산 frequency bins | 미구현 |
| FFT | DFT 계산 알고리즘 | DFT와 같은 결과 | 미구현 |

## 4. N과 샘플 수 M의 차이

N을 늘리면 근사에 사용하는 조화파 항이 많아집니다. M을 늘리면 적분과 그래프의
표본 간격 Δx=2L/(M-1)이 작아집니다. **M이 커져도 N이 같으면 추가 조화파가 생기지 않습니다.**
N이 높을수록 빠른 진동을 표현하므로 충분한 M이 필요하며, 입력 함수가 빠르게 진동하면
고정된 표본으로 계수를 정확하게 구하지 못할 수 있습니다. 표본 수를 바꿔 안정성을 확인하세요.

프로그램은 양 끝점을 포함한 `linspace`와 사다리꼴 적분을 사용합니다.

$$\int_{-L}^{L}g(x)dx\approx\sum_{j=0}^{M-2}\frac{g(x_j)+g(x_{j+1})}{2}(x_{j+1}-x_j).$$

표시 오차는 표본 오차이며 연속 구간 전체의 정확한 적분 오차나 최대 오차가 아닙니다.
$$\mathrm{MSE}=\frac1M\sum_j(f(x_j)-S_N(x_j))^2,\quad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}},\quad E_{\max}=\max_j|f(x_j)-S_N(x_j)|.$$

## 5. 대칭성과 Gibbs 현상

f(x)=x, L=π이면 기함수이므로 a0=0, a_n=0이고
$$b_n=\frac{2(-1)^{n+1}}{n}.$$
수치 적분에서는 0 대신 매우 작은 반올림 오차가 남을 수 있습니다.
짝함수 x², |x|에서는 b_n이 거의 0입니다. 대칭 자동 판별 기능은 아직 없습니다.

`sign(x)`는 0에서 불연속입니다. 불연속점 근처에서 부분합이 위아래로 진동하는 것이 Gibbs 현상입니다.
N이 커질수록 진동 영역은 좁아지지만 overshoot는 점프 높이의 약 9% 수준으로 남습니다.
좌우 극한이 -1,1이면 점프 높이는 2이므로 위쪽 peak는 약 1.18입니다.
불연속점에서는 적절한 조건 아래 좌우 극한의 평균으로 수렴합니다.
`x`나 `exp(x)`도 주기적으로 이어 붙이면 양 끝에서 불연속이 생길 수 있습니다.

## 6. 실행 및 개발 환경

현재 PC에서는 노트북 옆 `실행.bat`를 실행하세요. 노트북 커널은
`Fourier Visualizer (.venv)` 또는 `.venv/Scripts/python.exe`를 선택합니다.
노트북 자체는 소스 생성과 학습·검증용이며 전체 실행으로 GUI를 자동 실행하지 않습니다.

새 PC에서 프로젝트 폴더를 기준으로 설치하려면:

```powershell
python -m venv .venv
& ./.venv/Scripts/python.exe -m pip install -r ./fourier_visualizer/requirements.txt ipykernel
& ./.venv/Scripts/python.exe ./fourier_visualizer/desktop_app.py
```

웹 버전이 필요할 때만 `fourier_visualizer` 폴더에서 다음을 사용합니다.
```bash
pip install -r requirements.txt
streamlit run app.py
```

## 7. 파일 구조와 데이터 흐름

`입력 → 수식 검사 → 표본 생성 → 계수 적분 → 부분합 → 오차 → 그래프/표`

- `fourier_core.py`: 공통 수학 함수와 Figure 생성. Streamlit/Tkinter 상태에 의존하지 않습니다.
- `desktop_app.py`: Tkinter 창, 숫자 입력/슬라이더 동기화, 결과 표시, 이벤트 처리.
- `app.py`: 선택적으로 사용하는 Streamlit 화면. 동일한 공통 계산 함수를 가져옵니다.
- `requirements.txt`: 설치할 라이브러리. Tkinter는 Python 설치에 포함되어야 합니다.
- `실행.bat`: 현재 폴더의 가상환경으로 독립 창을 실행하는 Windows 진입점.

## 8. 최적화 내용과 한계

여러 N의 부분합은 각 N마다 1부터 다시 더하지 않고 최대 N까지 한 번 누적하여
필요한 차수의 배열만 복사합니다. 표본 수 M일 때 계산량은 O(MΣN)에서 O(M max N)으로 줄어듭니다.
스냅샷 배열은 선택 차수 개수 K에 대해 O(KM) 메모리가 필요합니다.
각 snapshot은 복사본이므로 뒤의 누적 연산에 의해 앞의 결과가 바뀌지 않습니다.

데스크톱 `FourierSession`은 마지막 함수 문자열, L, M에 대한 표본과 계수를 보관합니다.
같은 조건에서 이미 계산한 최대 차수 이하를 요청하면 적분을 생략합니다.
더 큰 차수를 요청하면 계수를 다시 계산하고, 조건이 바뀌면 새 표본으로 교체합니다.
전체 이력을 저장하지 않으므로 캐시가 무한히 커지지 않습니다.
수식 해석은 최대 32개 LRU 캐시로 반복 SymPy 해석을 줄입니다.

250ms 입력 지연 처리는 기존부터 적용된 기능입니다. 빠른 슬라이더 이동 중 예약 계산을 취소합니다.
그래프는 현재도 재생성하므로 화면 그리기 비용은 남아 있습니다.
이번 변경은 계산 중복 제거이며 전체 UI 속도가 일정 배수 빨라진다는 보장은 하지 않습니다.
큰 100×20000 삼각함수 행렬을 항상 만들지 않고 차수별 배열을 사용해 메모리 사용을 제한합니다.

## 9. 확장 계획

v0.3: 대칭 자동 판별, A_n=√(a_n²+b_n²) amplitude spectrum,
위상 관례를 정한 phase spectrum, 표본 표시, Gibbs 강조, 애니메이션, 예제 선택.
v0.4: DFT/FFT, sampling frequency, frequency bins, aliasing, 급수와 DFT 비교.
예를 들어 a cosθ+b sinθ=A cos(θ-φ) 관례라면 φ=atan2(b,a)이며 A=0의 위상은 별도 처리합니다.
'''
(project_directory / 'README.md').write_text(file_text, encoding='utf-8')


In [ ]:
launcher = r'''@echo off
cd /d "%~dp0"
if not exist ".venv\Scripts\pythonw.exe" (
    echo Python environment not found. Please create .venv first.
    pause
    exit /b 1
)
start "" ".venv\Scripts\pythonw.exe" "fourier_visualizer\desktop_app.py"
'''
(notebook_directory / '실행.bat').write_text(launcher, encoding='utf-8')


## 수학 및 최적화 검증

라이브러리가 설치된 `.venv` 커널에서 실행합니다.
해석적 x의 계수, 다중 부분합과 독립 공식의 일치, 캐시 재사용/무효화,
상수 함수와 sign(x)의 유한성을 검사합니다. 시간 측정 수치는 컴퓨터마다 달라집니다.


In [ ]:
import importlib
import numpy as np
sys.path.insert(0, str(project_directory.resolve()))
import fourier_core
core = importlib.reload(fourier_core)
x = np.linspace(-np.pi, np.pi, 20000)
a0, an, bn = core.calculate_fourier_coefficients(x, x, np.pi, 100)
n = np.arange(1, 101)
assert abs(a0) < 1e-12 and np.max(np.abs(an)) < 1e-12
assert np.allclose(bn, 2 * (-1.0)**(n+1) / n, atol=1e-5)
results = core.calculate_fourier_sums(x, np.pi, a0, an, bn, [0, 1, 10, 50, 100])
for order, actual in results.items():
    expected = np.full_like(x, a0)
    for k in range(1, order+1):
        expected += an[k-1]*np.cos(k*x) + bn[k-1]*np.sin(k*x)
    assert np.allclose(actual, expected, atol=1e-12)
assert core.calculate_fourier_sums(x, np.pi, a0, an, bn, []) == {}
session = core.FourierSession()
session.prepare('x', np.pi, 5000, 50)
saved = session.data
session.prepare('x', np.pi, 5000, 10)
assert session.data is saved
session.prepare('x', np.pi, 1234, 10)
assert len(session.data[1]) == 1234 and session.data is not saved
for text in ['3', 'sin(x)', 'cos(x)', 'x**2', 'exp(x)', 'abs(x)', 'sign(x)']:
    expression, grid, y, a0, an, bn = session.prepare(text, np.pi, 5000, 50)
    result = core.calculate_fourier_sum(grid, np.pi, a0, an, bn, 50)
    assert np.all(np.isfinite(result))
    if text == '3':
        assert np.allclose(result, 3)
print('수학 공식, 다중 부분합, 캐시, 필수 함수 검증 통과')
